# 11강 — 분석 결과 해석 및 시각화

10강에서 학습한 RandomForest 이탈 예측 모델의 성능 지표를 시각화하고, 결과를 해석해 인사이트 리포트를 작성합니다.

이 노트북은 10강과 동일한 데이터로 다시 시작합니다(독립 실행 가능).

In [ ]:
from pyspark.sql import SparkSession
import random

spark = SparkSession.builder.appName("result-interpretation").master("local[*]").getOrCreate()

random.seed(42)
rows = []
for i in range(2000):
    tenure = random.randint(1, 72)
    monthly_charge = round(random.uniform(20, 120), 2)
    support_calls = random.randint(0, 10)
    contract = random.choice(["month-to-month", "one-year", "two-year"])
    score = (-0.05*tenure) + (0.3*support_calls) + (10 if contract=="month-to-month" else 0) + random.gauss(0, 3)
    churn = 1 if score > 8 else 0
    rows.append((tenure, monthly_charge, support_calls, contract, churn))

df = spark.createDataFrame(rows, ["tenure", "monthly_charge", "support_calls", "contract", "churn"])
print("SparkSession OK:", spark.version)

## 1. 모델 재학습 (10강과 동일, RandomForest 고정 파라미터)

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

indexer = StringIndexer(inputCol="contract", outputCol="contract_idx")
encoder = OneHotEncoder(inputCols=["contract_idx"], outputCols=["contract_vec"])
assembler = VectorAssembler(
    inputCols=["tenure", "monthly_charge", "support_calls", "contract_vec"],
    outputCol="features",
)
rf = RandomForestClassifier(featuresCol="features", labelCol="churn", numTrees=30, maxDepth=6, seed=42)
pipe = Pipeline(stages=[indexer, encoder, assembler, rf])

train, test = df.randomSplit([0.8, 0.2], seed=42)
model = pipe.fit(train)
pred = model.transform(test)
print("예측 완료:", pred.count(), "건")

## 2. 혼동행렬 (Confusion Matrix) — 어떤 실수를 하고 있는가

In [ ]:
cm = (
    pred.groupBy("churn")
    .pivot("prediction", [0.0, 1.0])
    .count()
    .fillna(0)
    .orderBy("churn")
    .toPandas()
)
cm.columns = ["실제_이탈여부", "예측_잔류(0)", "예측_이탈(1)"]
print(cm)

import matplotlib.pyplot as plt
import numpy as np

matrix = cm[["예측_잔류(0)", "예측_이탈(1)"]].values
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(matrix, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["예측: 잔류", "예측: 이탈"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["실제: 잔류", "실제: 이탈"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=14)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

### 해석 포인트
- 왼쪽 아래(실제 이탈인데 잔류로 예측)가 **False Negative** — 이탈할 고객을 놓치는 것이라 비즈니스적으로 가장 아픈 실수입니다.
- 오른쪽 위(실제 잔류인데 이탈로 예측)는 **False Positive** — 불필요한 이탈 방지 캠페인 비용이 나가지만, 놓치는 것보다는 덜 치명적입니다.

## 3. ROC 곡선 — 임계값을 바꿔가며 본 성능

In [ ]:
from pyspark.mllib.evaluation import BinaryClassificationMetrics

scores_labels = pred.select("probability", "churn").rdd.map(
    lambda r: (float(r["probability"][1]), float(r["churn"]))
)
metrics = BinaryClassificationMetrics(scores_labels)
roc_points = metrics.roc().collect()

fpr = [p[0] for p in roc_points]
tpr = [p[1] for p in roc_points]

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"RandomForest (AUC={metrics.areaUnderROC:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="무작위 추측")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

print(f"AUC = {metrics.areaUnderROC:.3f}")

### 생각해 볼 질문
1. ROC 곡선이 대각선(무작위 추측)에 가까울수록 좋은 걸까요, 나쁜 걸까요?
2. AUC 0.95는 "항상 좋은 모델"을 의미할까요? 데이터 자체가 쉬운 문제라서 높게 나온 건 아닌지 어떻게 확인할 수 있을까요?

## 4. Feature Importance — 어떤 변수가 이탈을 가장 잘 설명하는가

In [ ]:
rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray()
feature_names = ["tenure", "monthly_charge", "support_calls",
                  "contract_one-year", "contract_two-year"]

imp_df = list(zip(feature_names, importances))
imp_df.sort(key=lambda x: x[1], reverse=True)

names = [x[0] for x in imp_df]
values = [x[1] for x in imp_df]

plt.figure(figsize=(6, 4))
plt.barh(names, values)
plt.xlabel("Feature Importance")
plt.title("RandomForest Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

for name, val in imp_df:
    print(f"{name:20s} {val:.3f}")

### 생각해 볼 질문
3. 가장 중요한 피처가 `contract`(계약 형태) 관련이라면, 이걸 비즈니스 액션(예: 프로모션)으로 어떻게 연결할 수 있을까요?
4. 이 중요도 순위가 실제 원인(인과관계)이라고 확신할 수 있을까요, 아니면 상관관계일 뿐일까요?

## 5. 인사이트 리포트 작성 (직접 채우세요)

아래 템플릿을 이 노트북의 결과(혼동행렬/ROC/Feature Importance)를 근거로
채우세요. 숫자를 나열하는 게 아니라 **"그래서 무엇을 해야 하는가"**로
끝나는 게 목표입니다.

---

### 이탈 예측 모델 분석 리포트

**1. 배경**
- 무엇을 예측하는 모델인가:
- 왜 이 예측이 비즈니스에 중요한가:

**2. 모델 성능 요약**
- AUC: ______  (모델이 이탈/잔류를 얼마나 잘 구분하는가)
- 가장 흔한 실수 유형(False Negative vs False Positive):

**3. 핵심 발견**
- 이탈에 가장 큰 영향을 주는 요인 1~2개:
- 그 요인이 왜 이탈과 관련 있을 것 같은지에 대한 가설:

**4. 권장 조치**
- 이 모델 결과를 바탕으로 무엇을 다르게 해야 하는가 (예: 특정 계약 유형
  고객 대상 리텐션 캠페인):
- 이 모델을 실무에 쓰기 전에 추가로 확인해야 할 것:

---


In [ ]:
spark.stop()